# Anti-UAV YOLO26s Training — Full Run (100% data, 100 epochs)
**Model:** yolo26s | **Hardware:** Colab T4 | **Dataset:** Full merged (Bird/Drone/UAV)

**Estimated time:** ~6-8 hours | **Checkpoints saved to Drive every 10 min**

**Before running:**
1. Upload backup_merged_dataset.tar.gz to this account's Google Drive
2. Runtime → Change runtime type → T4 GPU
3. Runtime → Run all

In [ ]:
# Download dataset from shared Google Drive link
import os, subprocess

# Paste your shared Drive link here
SHARED_LINK = 'PASTE_YOUR_SHARED_DRIVE_LINK_HERE'

# Extract file ID from the link
if 'id=' in SHARED_LINK:
    file_id = SHARED_LINK.split('id=')[1].split('&')[0]
elif '/d/' in SHARED_LINK:
    file_id = SHARED_LINK.split('/d/')[1].split('/')[0]
else:
    raise ValueError('Could not extract file ID from link')

print(f'File ID: {file_id}')
tar_path = '/content/backup_merged_dataset.tar.gz'

# Download using gdown
subprocess.run(['pip', 'install', '-q', 'gdown'], check=True)
import gdown
gdown.download(id=file_id, output=tar_path, quiet=False)
print(f'Downloaded: {tar_path} ({os.path.getsize(tar_path)/1e9:.2f} GB)')


In [ ]:
# Mount Drive for checkpoint saving + extract dataset
from google.colab import drive
import tarfile, os, yaml, subprocess
drive.mount('/content/drive')
result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], capture_output=True, text=True)
print('GPU:', result.stdout.strip())

extract_dir = '/content/dataset'
os.makedirs(extract_dir, exist_ok=True)
print('Extracting dataset (~2 min)...')
with tarfile.open(tar_path) as tf:
    tf.extractall(extract_dir)

data_yaml = os.path.join(extract_dir, 'merged_dataset', 'data.yaml')
base = os.path.join(extract_dir, 'merged_dataset')
with open(data_yaml) as f:
    cfg = yaml.safe_load(f)
cfg['path'] = base
cfg['train'] = os.path.join(base, 'train', 'images')
cfg['val'] = os.path.join(base, 'val', 'images')
cfg['test'] = os.path.join(base, 'test', 'images')
with open(data_yaml, 'w') as f:
    yaml.dump(cfg, f)

for split in ['train', 'val', 'test']:
    n = len(os.listdir(cfg[split]))
    print(f'  {split}: {n} images')


In [ ]:
# Install/upgrade ultralytics and verify version
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', 'ultralytics>=8.4.0'], check=True)
import ultralytics
print(f'ultralytics {ultralytics.__version__}')
assert tuple(int(x) for x in ultralytics.__version__.split('.')[:2]) >= (8, 4), \
    f'Need ultralytics>=8.4.0, got {ultralytics.__version__}. Restart runtime and re-run.'
print('Version OK — YOLO26 supported')


In [ ]:
# Run training — yolo26s FULL run with Drive checkpoint backup
import threading, time, shutil, os
from ultralytics import YOLO

DRIVE_BACKUP = '/content/drive/MyDrive/anti_uav_checkpoints_run2'
os.makedirs(DRIVE_BACKUP, exist_ok=True)
WEIGHTS_DIR = '/content/runs/anti_uav_run2_yolo26s_full/weights'
stop_backup = threading.Event()

def backup_to_drive():
    while not stop_backup.is_set():
        time.sleep(600)  # every 10 minutes
        if os.path.exists(WEIGHTS_DIR):
            for f in os.listdir(WEIGHTS_DIR):
                src = os.path.join(WEIGHTS_DIR, f)
                dst = os.path.join(DRIVE_BACKUP, f)
                try:
                    shutil.copy2(src, dst)
                    print(f'[backup] {f} → Drive')
                except Exception as e:
                    print(f'[backup] Failed {f}: {e}')

backup_thread = threading.Thread(target=backup_to_drive, daemon=True)
backup_thread.start()
print('Checkpoint backup thread started (saves to Drive every 10 min)')

model = YOLO('yolo26s.pt')
results = model.train(
    data=data_yaml,
    imgsz=640,
    batch=32,
    epochs=100,
    patience=30,
    fraction=1.0,
    optimizer='MuSGD',
    lr0=0.01,
    weight_decay=0.0005,
    amp=True,
    device='0',
    save_period=10,
    mosaic=1.0,
    mixup=0.05,
    copy_paste=0.5,
    hsv_h=0.02,
    hsv_s=0.7,
    hsv_v=0.5,
    degrees=20.0,
    translate=0.15,
    scale=0.8,
    flipud=0.3,
    fliplr=0.5,
    project='/content/runs',
    name='anti_uav_run2_yolo26s_full',
)
stop_backup.set()
print(f'Training complete — results: {results.save_dir}')


In [ ]:
# Archive full run and save to Drive
import zipfile, os, shutil
runs_dir = '/content/runs/anti_uav_run2_yolo26s_full'
archive_drive = '/content/drive/MyDrive/anti_uav_run2_yolo26s_full.zip'

print('Archiving full run directory...')
with zipfile.ZipFile(archive_drive, 'w', zipfile.ZIP_DEFLATED) as zf:
    for root, dirs, files in os.walk(runs_dir):
        for file in files:
            filepath = os.path.join(root, file)
            arcname = os.path.relpath(filepath, '/content')
            zf.write(filepath, arcname)
            print(f'  {arcname} ({os.path.getsize(filepath)/1e6:.1f} MB)')

print(f'Saved to Drive: {archive_drive} ({os.path.getsize(archive_drive)/1e6:.1f} MB)')
print('Done')
